# Material de aula - FACENS - Aprendizado de Máquina


## Perceptron e Regra de Aprendizado de Hebb - Laboratório 02

In [2]:
#CÉLULA PERCEPTRON-LIB-01
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [3]:
#CÉLULA PERCEPTRON-LIB-02
#Função para carregamento de dados de treinamento
def loadDataFile(filename) :
    '''
    Esta função faz o carregamento de dados de um arquivo separado por virgulas (csv)
    
    Parametros
    -----------
    filename -> nome do arquivo de dados em formato csv
    
    
    Retorno
    -----------  
    dataframe contendo os dados do arquivo formatados numericamente.
    '''
    
    dfFile = pd.read_csv(filename, sep=';')
    
    #Os numeros estao com virgula, troque para ponto
    dfFile = dfFile.replace(regex=',', value='.')
    
    #Converta para numeros
    #Verifique se a coluna existe antes de manipulá-la    
    if ('x1' in dfFile.columns) :
        dfFile['x1']  = pd.to_numeric(dfFile['x1'])

    if ('x2' in dfFile.columns) :
        dfFile['x2']  = pd.to_numeric(dfFile['x2'])

    if ('x3' in dfFile.columns) :
        dfFile['x3']  = pd.to_numeric(dfFile['x3'])

    if ('d' in dfFile.columns) :
        dfFile['d']  = pd.to_numeric(dfFile['d'])
        
    return dfFile


#Validação
dfTrainingFile = loadDataFile('Lab01_training.csv')
print(dfTrainingFile.dtypes)

dfOperationFile = loadDataFile('Lab01_operation.csv')
print(dfOperationFile.dtypes)

Amostra      int64
x1         float64
x2         float64
x3         float64
d            int64
dtype: object
x1    float64
x2    float64
x3    float64
dtype: object


In [4]:
dfTrainingFile

,Amostra,x1,x2,x3,d
0,1,-0.6508,0.1097,4.0009,-1
1,2,-1.4492,0.8896,4.4005,-1
2,3,2.0850,0.6876,12.0710,-1
3,4,0.2626,1.1476,7.7985,1
4,5,0.6418,1.0234,7.0427,1
5,6,0.2569,0.6730,8.3265,-1
6,7,1.1155,0.6043,7.4446,1
7,8,0.0914,0.3399,7.0677,-1
8,9,0.0121,0.5256,4.6316,1
9,10,-0.0429,0.4660,5.4323,1


In [5]:
dfOperationFile

,x1,x2,x3
0,-0.3565,0.0620,5.9891
1,-0.7842,1.1267,5.5912
2,0.3012,0.5611,5.8234
3,0.7757,1.0648,8.0677
4,0.1570,0.8028,6.3040
5,-0.7014,1.0316,3.6005
6,0.3748,0.1536,6.1537
7,-0.6920,0.9404,4.4058
8,-1.3970,0.7141,4.9263
9,-1.8842,-0.2805,1.2548


# 1 - Criando a função de escala - Parte 1

Utilizando Jupyter e NumPy: para a função de variável contínua apresentada, construa uma função para o cálculo da escala dos dados para o domínio da função de ativação.

Assinatura: **def tales(x1, x1_min, x1_max, x2_min, x2_max) :**

### esta função faz o escalamento de 1 amostra x1, que pertence ao intervalo [x1_min, x1_max],
### dentro do intervalo [x2_min, x2_max]
    
**Retorno da Função:** valor dentro do domínio dado

In [6]:
#CÉLULA PERCEPTRON-LIB-03
# Função para cálculo de normalização de uma amostra
def tales(x1, x1_min, x1_max, x2_min, x2_max) :
    '''
    Esta função faz o escalamento de 1 amostra x1, que pertence ao intervalo [x1_min, x1_max], 
    dentro do intervalo [x2_min, x2_max]
    
    Parametros
    -----------
    x1     -> valor a ser normalizado
    x1_min -> valor minimo atingido por x1
    x1_max -> valor maximo atingido por x1
    x2_min -> valor minimo da escala de destino
    x2_max -> valor maximo da escala de destino    
    
    Retorno
    -----------  
    valor de x1, que pertence ao intervalo [x1_min, x1_max], projetado dentro do intervalo [x2_min, x2_max]
    '''
   
    x2 = ((x1- x1_min) * (x2_max - x2_min)) / (x1_max - x1_min) + x2_min
    
    return x2

In [9]:
#Teste
tales(5, -10, 10, -1, 1)


0.5

## Validação
``
tales(5, -10, 10, -1, 1)
``
Valor esperado: 0.5



``
tales(-5, -10, 10, -1, 1)
``
Valor esperado: -0.5


``
tales(-10, -10, 10, -1, 1)
``
Valor esperado: -1.0


``
tales(10, -10, 10, -1, 1)
``
Valor esperado: 1.0

# 2 - Criando a função de escala - Parte 2

Utilizando Jupyter e NumPy: Utilizando Jupyter e NumPy: para a função de variável contínua apresentada, construa uma função para o cálculo da escala do vetor de dados para o domínio da função de ativação.

Assinatura: **def scale(x2_min, x2_max, sampleArray) :**

### esta função faz o escalamento de 1 array de dados que para dentro do intervalo [x2_min, x2_max]
    
**Retorno da Função:** valor dentro do domínio dado

**NOTA**: Não sabemos os limites da escala do array, temos que descobrir!

In [10]:
#CÉLULA PERCEPTRON-LIB-04
def scale(x2_min, x2_max, sampleArray) :    
    '''
    Esta função faz o escalamento de 1 vetor de amostras para dentro do intervalo [x2_min, x2_max]
    Ela retorna um vetor com os dados normalizados dentro do dominio dado
    
    Parametros
    -----------
    x2_min      -> valor minimo da escala de destino
    x2_max      -> valor maximo da escala de destino
    sampleArray -> vetor a ser normalizado   
    
    Retorno
    -----------  
    vetor sampleArray projetado dentro do intervalo [x2_min, x2_max]   
    '''
    sampleArrayNorm = tales(sampleArray, min(sampleArray),max(sampleArray),x2_min,x2_max)    
        
    return sampleArrayNorm

In [11]:
#Teste
v = np.array([5, -5, 10, -10, 0, 3, -3, 4, -4, 7])
scale(-1, 1, v)

array([ 0.5, -0.5,  1. , -1. ,  0. ,  0.3, -0.3,  0.4, -0.4,  0.7])

## Validação
``
v = np.array([5, -5, 10, -10, 0, 3, -3, 4, -4, 7])
scale(-1, 1, v)
``

**Valor esperado:**
    [0.5, -0.5, 1.0, -1.0, 0.0, 0.3, -0.3, 0.39, -0.4, 0.7]

# 3 - Teste sua função de escala com os dados do problema

Faça a carga dos dados e verifique se consegue fazer o escalamento das colunas de dados de treinamento e operação